# SafeFundus — Notebook 06B: Swin-Tiny Weighted-BCE Multi-Seed Classifier

## Purpose

Notebook 06 trained the selected **Swin-Tiny** backbone with **Asymmetric Loss (ASL)** across three seeds.

This notebook performs the required controlled comparison using **weighted binary cross-entropy (Weighted BCE)** across the exact same seeds:

- **42**
- **123**
- **2026**

The goal is to choose the final SafeFundus classifier loss using primarily:

\[
\textbf{mean validation mAP across 3 seeds}
\]

rather than relying on a single run or selecting based on the test set.

## Fair comparison with Notebook 06

Everything stays the same except the loss:

- same Swin-Tiny backbone
- same ImageNet pretrained weights
- same RFMiD known/unknown protocol
- same training/validation/test construction
- same 224 × 224 preprocessing
- same augmentation
- same optimizer
- same learning rate
- same weight decay
- same dropout
- same early stopping
- same validation-mAP model selection
- same per-class threshold search
- same seeds

### Only changed component

Notebook 06:

\[
\text{Asymmetric Loss}
\]

Notebook 06B:

\[
\text{Weighted BCEWithLogitsLoss}
\]

After this experiment we will freeze the better loss and move to JointEnergy without retraining the classifier.


In [1]:
# ============================================================
# 1. Imports
# ============================================================

from pathlib import Path
import os
import json
import random
import time
import warnings
import shutil
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import Swin_T_Weights
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


# 2. Configuration

The optimization recipe is intentionally matched to Notebook 06 and the original Swin-Tiny backbone-screening run.


In [2]:
# ============================================================
# 2. Configuration
# ============================================================

CONFIG = {
    "seeds": [42, 123, 2026],

    "image_size": 224,
    "batch_size": 32,
    "epochs": 35,

    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "dropout": 0.30,

    "num_workers": 2,
    "early_stopping_patience": 7,
    "lr_patience": 3,

    "threshold_grid_start": 0.05,
    "threshold_grid_end": 0.95,
    "threshold_grid_step": 0.01,

    # Match the original backbone-screening BCE setup.
    "pos_weight_cap": 20.0,

    "model_name": "swin_tiny",
    "loss_name": "weighted_bce",
}

DATA_ROOT = Path(
    "/kaggle/input/datasets/ozlemhakdagli/"
    "retinal-fundus-multi-disease-image-dataset-rfmid"
)

TRAIN_DIR = DATA_ROOT / "Training_set"
VAL_DIR   = DATA_ROOT / "Validation_set"
TEST_DIR  = DATA_ROOT / "Test_set"

TRAIN_CSV = TRAIN_DIR / "RFMiD_Training_Labels.csv"
VAL_CSV   = VAL_DIR   / "RFMiD_Validation_Labels.csv"
TEST_CSV  = TEST_DIR  / "RFMiD_Testing_Labels.csv"

NOTEBOOK_STEM = "06B_SwinTiny_WeightedBCE_MultiSeed_Classifier"
OUTPUT_DIR = Path("/kaggle/working") / NOTEBOOK_STEM

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"
PRED_DIR = OUTPUT_DIR / "predictions"

for p in [
    OUTPUT_DIR,
    CHECKPOINT_DIR,
    TABLE_DIR,
    FIG_DIR,
    PRED_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(json.dumps(CONFIG, indent=2))
print("\nDEVICE:", DEVICE)
print("OUTPUT:", OUTPUT_DIR)


{
  "seeds": [
    42,
    123,
    2026
  ],
  "image_size": 224,
  "batch_size": 32,
  "epochs": 35,
  "learning_rate": 0.0001,
  "weight_decay": 0.0001,
  "dropout": 0.3,
  "num_workers": 2,
  "early_stopping_patience": 7,
  "lr_patience": 3,
  "threshold_grid_start": 0.05,
  "threshold_grid_end": 0.95,
  "threshold_grid_step": 0.01,
  "pos_weight_cap": 20.0,
  "model_name": "swin_tiny",
  "loss_name": "weighted_bce"
}

DEVICE: cuda
OUTPUT: /kaggle/working/06B_SwinTiny_WeightedBCE_MultiSeed_Classifier


# 3. Frozen SafeFundus Disease Protocol


In [3]:
# ============================================================
# 3. Disease protocol
# ============================================================

KNOWN_CLASSES = [
    "DR", "MH", "ODC", "TSLN", "DN", "MYA", "ARMD",
    "BRVO", "ODP", "ODE", "LS", "RS", "CSR", "CRS"
]

INTERMEDIATE_CLASSES = [
    "CRVO",
    "RPEC",
]

print("Known classes:", len(KNOWN_CLASSES))
print(KNOWN_CLASSES)
print("\nIntermediate:", INTERMEDIATE_CLASSES)


Known classes: 14
['DR', 'MH', 'ODC', 'TSLN', 'DN', 'MYA', 'ARMD', 'BRVO', 'ODP', 'ODE', 'LS', 'RS', 'CSR', 'CRS']

Intermediate: ['CRVO', 'RPEC']


# 4. Load RFMiD and Reconstruct the Frozen Groups


In [4]:
# ============================================================
# 4. Load labels
# ============================================================

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

ID_COL = "ID"
RISK_COL = "Disease_Risk"

assert ID_COL in train_df.columns
assert RISK_COL in train_df.columns

DISEASE_COLS = [
    c for c in train_df.columns
    if c not in {ID_COL, RISK_COL}
]

missing_known = [
    c for c in KNOWN_CLASSES
    if c not in DISEASE_COLS
]

assert not missing_known, (
    f"Missing known classes: {missing_known}"
)

UNKNOWN_CLASSES = [
    c for c in DISEASE_COLS
    if c not in KNOWN_CLASSES
    and c not in INTERMEDIATE_CLASSES
]

def assign_group(row):
    positive = {
        c for c in DISEASE_COLS
        if int(row[c]) == 1
    }

    if len(positive) == 0:
        return "Normal"

    if positive & set(INTERMEDIATE_CLASSES):
        return "Contains_Intermediate"

    has_known = bool(
        positive & set(KNOWN_CLASSES)
    )

    has_unknown = bool(
        positive & set(UNKNOWN_CLASSES)
    )

    if has_known and has_unknown:
        return "Mixed_Known_Unknown"

    if has_known:
        return "Known_Only"

    if has_unknown:
        return "Unknown_Only"

    return "Other"

for df in [train_df, val_df, test_df]:
    df["Group"] = df.apply(
        assign_group,
        axis=1,
    )

for name, df in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df),
]:
    print(f"\n{name}")
    display(
        df["Group"]
        .value_counts()
        .rename_axis("Group")
        .reset_index(name="Count")
    )



TRAIN


,Group,Count
0,Known_Only,1312
1,Normal,401
2,Mixed_Known_Unknown,82
3,Unknown_Only,75
4,Contains_Intermediate,50



VALIDATION


,Group,Count
0,Known_Only,426
1,Normal,134
2,Unknown_Only,34
3,Mixed_Known_Unknown,32
4,Contains_Intermediate,14



TEST


,Group,Count
0,Known_Only,437
1,Normal,134
2,Unknown_Only,32
3,Mixed_Known_Unknown,24
4,Contains_Intermediate,13


# 5. Final Experimental Splits


In [5]:
# ============================================================
# 5. Experimental splits
# ============================================================

train_exp = train_df[
    train_df["Group"].isin(
        ["Normal", "Known_Only"]
    )
].copy()

val_exp = val_df[
    val_df["Group"].isin(
        ["Normal", "Known_Only"]
    )
].copy()

test_familiar = test_df[
    test_df["Group"].isin(
        ["Normal", "Known_Only"]
    )
].copy()

test_unknown = test_df[
    test_df["Group"] == "Unknown_Only"
].copy()

test_mixed = test_df[
    test_df["Group"] == "Mixed_Known_Unknown"
].copy()

test_intermediate = test_df[
    test_df["Group"] == "Contains_Intermediate"
].copy()

print("Train             :", len(train_exp))
print("Validation        :", len(val_exp))
print("Familiar test     :", len(test_familiar))
print("Unknown-only test :", len(test_unknown))
print("Mixed test        :", len(test_mixed))
print("Intermediate test :", len(test_intermediate))

assert len(train_exp) == 1713
assert len(val_exp) == 560
assert len(test_familiar) == 571
assert len(test_unknown) == 32
assert len(test_mixed) == 24

print("\n✓ Frozen SafeFundus protocol verified.")


Train             : 1713
Validation        : 560
Familiar test     : 571
Unknown-only test : 32
Mixed test        : 24
Intermediate test : 13

✓ Frozen SafeFundus protocol verified.


# 6. Verify Image Paths


In [6]:
# ============================================================
# 6. Image paths
# ============================================================

def normalize_id(x):
    return str(int(float(x)))

def add_image_paths(df, folder):
    out = df.copy()

    out["image_path"] = out[ID_COL].apply(
        lambda x: str(
            folder /
            f"{normalize_id(x)}.png"
        )
    )

    missing = [
        p for p in out["image_path"]
        if not Path(p).exists()
    ]

    if missing:
        raise FileNotFoundError(
            f"{len(missing)} images missing. "
            f"Example: {missing[:3]}"
        )

    return out

train_exp = add_image_paths(
    train_exp,
    TRAIN_DIR,
)

val_exp = add_image_paths(
    val_exp,
    VAL_DIR,
)

test_familiar = add_image_paths(
    test_familiar,
    TEST_DIR,
)

test_unknown = add_image_paths(
    test_unknown,
    TEST_DIR,
)

test_mixed = add_image_paths(
    test_mixed,
    TEST_DIR,
)

test_intermediate = add_image_paths(
    test_intermediate,
    TEST_DIR,
)

print("All image paths verified.")


All image paths verified.


# 7. Image Preprocessing

This is kept identical to Notebook 06.

### Training

- resize to 224 × 224
- horizontal flip
- ±10° rotation
- mild color jitter
- ImageNet normalization

### Validation/Test

- deterministic resize
- ImageNet normalization


In [7]:
# ============================================================
# 7. Transforms
# ============================================================

weights = Swin_T_Weights.IMAGENET1K_V1

imagenet_mean = weights.transforms().mean
imagenet_std = weights.transforms().std

train_transform = transforms.Compose([
    transforms.Resize(
        (
            CONFIG["image_size"],
            CONFIG["image_size"],
        )
    ),
    transforms.RandomHorizontalFlip(
        p=0.5
    ),
    transforms.RandomRotation(
        degrees=10
    ),
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.08,
        hue=0.02,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std,
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize(
        (
            CONFIG["image_size"],
            CONFIG["image_size"],
        )
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std,
    ),
])

print("Mean:", imagenet_mean)
print("Std :", imagenet_std)


Mean: [0.485, 0.456, 0.406]
Std : [0.229, 0.224, 0.225]


# 8. PyTorch Dataset


In [8]:
# ============================================================
# 8. Dataset
# ============================================================

class RFMiDDataset(Dataset):
    def __init__(
        self,
        dataframe,
        transform=None,
    ):
        self.df = (
            dataframe
            .reset_index(drop=True)
        )

        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        target = torch.tensor(
            row[KNOWN_CLASSES]
            .astype(np.float32)
            .values,
            dtype=torch.float32,
        )

        return {
            "image": image,
            "target": target,
            "id": normalize_id(
                row[ID_COL]
            ),
            "group": row["Group"],
        }


# 9. Weighted BCE Class Weights

The original Swin-Tiny screening experiment used:

\[
\text{pos\_weight}_c
=
\frac{N_{\text{negative},c}}
{N_{\text{positive},c}}
\]

with a maximum weight of 20.

The exact same weighting rule is used for every seed.


In [9]:
# ============================================================
# 9. Weighted BCE
# ============================================================

positive_counts = (
    train_exp[KNOWN_CLASSES]
    .sum(axis=0)
    .astype(float)
)

negative_counts = (
    len(train_exp)
    - positive_counts
)

pos_weight = (
    negative_counts
    / np.maximum(
        positive_counts,
        1.0,
    )
)

pos_weight = np.minimum(
    pos_weight,
    CONFIG["pos_weight_cap"],
)

class_balance_df = pd.DataFrame({
    "Disease":
        KNOWN_CLASSES,
    "Positive_Train":
        positive_counts
        .values
        .astype(int),
    "Negative_Train":
        negative_counts
        .values
        .astype(int),
    "Pos_Weight":
        pos_weight.values,
})

display(class_balance_df)

class_balance_df.to_csv(
    TABLE_DIR /
    "class_balance.csv",
    index=False,
)

POS_WEIGHT_TENSOR = torch.tensor(
    pos_weight.values,
    dtype=torch.float32,
    device=DEVICE,
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=POS_WEIGHT_TENSOR
)

print("Loss:", criterion)


,Disease,Positive_Train,Negative_Train,Pos_Weight
0,DR,358,1355,3.784916
1,MH,306,1407,4.598039
2,ODC,261,1452,5.563218
3,TSLN,174,1539,8.844828
4,DN,126,1587,12.595238
5,MYA,99,1614,16.303030
6,ARMD,92,1621,17.619565
7,BRVO,71,1642,20.000000
8,ODP,52,1661,20.000000
9,ODE,53,1660,20.000000


Loss: BCEWithLogitsLoss()


# 10. Build Swin-Tiny

The classifier head is:

\[
\text{Swin features}
\rightarrow
Dropout(0.30)
\rightarrow
14\text{ logits}
\]

The dropout remains in the model because later SafeFundus notebooks will use the same checkpoints for **MC Dropout inference**.


In [10]:
# ============================================================
# 10. Model
# ============================================================

def build_swin_tiny(
    num_classes,
    dropout=0.30,
):
    try:
        model = models.swin_t(
            weights=
            Swin_T_Weights.IMAGENET1K_V1
        )

        print(
            "Loaded ImageNet pretrained "
            "Swin-Tiny weights."
        )

    except Exception as e:
        print(
            "WARNING: pretrained weights "
            "could not be loaded."
        )
        print("Reason:", e)

        print(
            "\nFor the real experiment, "
            "enable Kaggle Internet or "
            "provide pretrained weights."
        )

        model = models.swin_t(
            weights=None
        )

    in_features = (
        model.head.in_features
    )

    model.head = nn.Sequential(
        nn.Dropout(
            p=dropout
        ),
        nn.Linear(
            in_features,
            num_classes,
        ),
    )

    return model


# 11. Metrics


In [11]:
# ============================================================
# 11. Metrics
# ============================================================

def safe_macro_auroc(
    y_true,
    y_prob,
):
    values = []

    for c in range(
        y_true.shape[1]
    ):
        if len(
            np.unique(
                y_true[:, c]
            )
        ) < 2:
            continue

        values.append(
            roc_auc_score(
                y_true[:, c],
                y_prob[:, c],
            )
        )

    return (
        float(np.mean(values))
        if values
        else np.nan
    )


def safe_map(
    y_true,
    y_prob,
):
    values = []

    for c in range(
        y_true.shape[1]
    ):
        if (
            y_true[:, c].sum()
            == 0
        ):
            continue

        values.append(
            average_precision_score(
                y_true[:, c],
                y_prob[:, c],
            )
        )

    return (
        float(np.mean(values))
        if values
        else np.nan
    )


def compute_epoch_metrics(
    y_true,
    y_prob,
    threshold=0.5,
):
    y_pred = (
        y_prob >= threshold
    ).astype(int)

    return {
        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            ),
        "micro_f1":
            f1_score(
                y_true,
                y_pred,
                average="micro",
                zero_division=0,
            ),
        "mAP":
            safe_map(
                y_true,
                y_prob,
            ),
        "macro_auroc":
            safe_macro_auroc(
                y_true,
                y_prob,
            ),
    }


# 12. Reproducibility and DataLoader Factory

Every seed starts from a fresh model and fresh deterministic random state.


In [12]:
# ============================================================
# 12. Reproducibility / loaders
# ============================================================

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)

    os.environ[
        "PYTHONHASHSEED"
    ] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = (
        torch.initial_seed()
        % (2**32)
    )

    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loaders(seed):
    train_dataset = RFMiDDataset(
        train_exp,
        train_transform,
    )

    val_dataset = RFMiDDataset(
        val_exp,
        eval_transform,
    )

    familiar_dataset = RFMiDDataset(
        test_familiar,
        eval_transform,
    )

    unknown_dataset = RFMiDDataset(
        test_unknown,
        eval_transform,
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    common = {
        "batch_size":
            CONFIG["batch_size"],
        "num_workers":
            CONFIG["num_workers"],
        "pin_memory":
            True,
        "persistent_workers":
            CONFIG["num_workers"] > 0,
    }

    train_loader = DataLoader(
        train_dataset,
        shuffle=True,
        worker_init_fn=seed_worker,
        generator=generator,
        **common,
    )

    val_loader = DataLoader(
        val_dataset,
        shuffle=False,
        **common,
    )

    familiar_loader = DataLoader(
        familiar_dataset,
        shuffle=False,
        **common,
    )

    unknown_loader = DataLoader(
        unknown_dataset,
        shuffle=False,
        **common,
    )

    return (
        train_loader,
        val_loader,
        familiar_loader,
        unknown_loader,
    )


# 13. Training and Evaluation Functions


In [13]:
# ============================================================
# 13. Train / evaluation functions
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
):
    model.train()

    running_loss = 0.0
    all_targets = []
    all_probs = []

    pbar = tqdm(
        loader,
        desc="Train",
        leave=False,
    )

    for batch in pbar:
        images = batch["image"].to(
            DEVICE,
            non_blocking=True,
        )

        targets = batch["target"].to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.cuda.amp.autocast(
            enabled=(
                DEVICE.type == "cuda"
            )
        ):
            logits = model(images)

            loss = criterion(
                logits,
                targets,
            )

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        probs = torch.sigmoid(
            logits
        )

        running_loss += (
            loss.item()
            * images.size(0)
        )

        all_targets.append(
            targets
            .detach()
            .cpu()
            .numpy()
        )

        all_probs.append(
            probs
            .detach()
            .cpu()
            .numpy()
        )

        pbar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    y_true = np.concatenate(
        all_targets
    )

    y_prob = np.concatenate(
        all_probs
    )

    metrics = compute_epoch_metrics(
        y_true,
        y_prob,
    )

    metrics["loss"] = (
        running_loss
        / len(loader.dataset)
    )

    return metrics


@torch.no_grad()
def evaluate_loader(
    model,
    loader,
    calculate_loss=False,
):
    model.eval()

    running_loss = 0.0
    all_targets = []
    all_probs = []
    all_logits = []
    all_ids = []
    all_groups = []

    for batch in tqdm(
        loader,
        desc="Evaluate",
        leave=False,
    ):
        images = batch["image"].to(
            DEVICE,
            non_blocking=True,
        )

        targets = batch["target"].to(
            DEVICE,
            non_blocking=True,
        )

        with torch.cuda.amp.autocast(
            enabled=(
                DEVICE.type == "cuda"
            )
        ):
            logits = model(
                images
            )

            if calculate_loss:
                loss = criterion(
                    logits,
                    targets,
                )

                running_loss += (
                    loss.item()
                    * images.size(0)
                )

        probs = torch.sigmoid(
            logits
        )

        all_targets.append(
            targets.cpu().numpy()
        )

        all_probs.append(
            probs.cpu().numpy()
        )

        all_logits.append(
            logits.cpu().numpy()
        )

        all_ids.extend(
            batch["id"]
        )

        all_groups.extend(
            batch["group"]
        )

    y_true = np.concatenate(
        all_targets
    )

    y_prob = np.concatenate(
        all_probs
    )

    logits = np.concatenate(
        all_logits
    )

    metrics = compute_epoch_metrics(
        y_true,
        y_prob,
    )

    if calculate_loss:
        metrics["loss"] = (
            running_loss
            / len(loader.dataset)
        )

    return {
        "metrics":
            metrics,
        "targets":
            y_true,
        "probs":
            y_prob,
        "logits":
            logits,
        "ids":
            all_ids,
        "groups":
            all_groups,
    }


# 14. Validation Threshold Optimization

Each seed receives its own validation-only thresholds:

\[
\tau_c
=
\arg\max_{\tau}
F1_c(\tau)
\]

No threshold is selected using RFMiD test labels.


In [14]:
# ============================================================
# 14. Threshold optimization
# ============================================================

THRESHOLD_GRID = np.arange(
    CONFIG[
        "threshold_grid_start"
    ],
    CONFIG[
        "threshold_grid_end"
    ] + 1e-9,
    CONFIG[
        "threshold_grid_step"
    ],
)

def optimize_thresholds(
    y_true,
    y_prob,
):
    thresholds = []

    for c in range(
        len(KNOWN_CLASSES)
    ):
        if (
            y_true[:, c].sum()
            == 0
        ):
            thresholds.append(
                0.5
            )
            continue

        best_threshold = 0.5
        best_f1 = -1.0

        for threshold in (
            THRESHOLD_GRID
        ):
            pred = (
                y_prob[:, c]
                >= threshold
            ).astype(int)

            score = f1_score(
                y_true[:, c],
                pred,
                zero_division=0,
            )

            if score > best_f1:
                best_f1 = score
                best_threshold = float(
                    threshold
                )

        thresholds.append(
            best_threshold
        )

    return np.asarray(
        thresholds,
        dtype=np.float32,
    )


# 15. Per-Class Test Metrics


In [15]:
# ============================================================
# 15. Per-class metrics
# ============================================================

def build_per_class_metrics(
    y_true,
    y_prob,
    thresholds,
):
    y_pred = (
        y_prob
        >= thresholds[None, :]
    ).astype(int)

    rows = []

    for c, disease in enumerate(
        KNOWN_CLASSES
    ):
        yt = y_true[:, c]
        yp = y_pred[:, c]
        probability = y_prob[:, c]

        tn, fp, fn, tp = (
            confusion_matrix(
                yt,
                yp,
                labels=[0, 1],
            ).ravel()
        )

        specificity = (
            tn / (tn + fp)
            if (tn + fp) > 0
            else np.nan
        )

        auroc = (
            roc_auc_score(
                yt,
                probability,
            )
            if len(np.unique(yt)) == 2
            else np.nan
        )

        ap = (
            average_precision_score(
                yt,
                probability,
            )
            if yt.sum() > 0
            else np.nan
        )

        rows.append({
            "Disease":
                disease,
            "Positive_Test":
                int(yt.sum()),
            "Threshold":
                float(
                    thresholds[c]
                ),
            "Precision":
                precision_score(
                    yt,
                    yp,
                    zero_division=0,
                ),
            "Sensitivity_Recall":
                recall_score(
                    yt,
                    yp,
                    zero_division=0,
                ),
            "Specificity":
                specificity,
            "F1":
                f1_score(
                    yt,
                    yp,
                    zero_division=0,
                ),
            "AUROC":
                auroc,
            "Average_Precision":
                ap,
            "TP":
                int(tp),
            "FP":
                int(fp),
            "TN":
                int(tn),
            "FN":
                int(fn),
        })

    return pd.DataFrame(
        rows
    )


# 16. Multi-Seed Weighted-BCE Training

For each seed:

1. reset random states;
2. create fresh data loaders;
3. initialize a fresh ImageNet-pretrained Swin-Tiny;
4. train with weighted BCE;
5. select the best checkpoint by validation mAP;
6. optimize class thresholds on validation only;
7. save validation/familiar/unknown logits;
8. evaluate classification and rare-to-normal safety;
9. preserve the checkpoint for JointEnergy and MC Dropout.


In [16]:
# ============================================================
# 16. Multi-seed training
# ============================================================

all_seed_summaries = []

for seed in CONFIG["seeds"]:

    print("\n")
    print("#" * 78)
    print(
        f"STARTING SEED {seed}"
    )
    print("#" * 78)

    seed_everything(seed)

    (
        train_loader,
        val_loader,
        familiar_loader,
        unknown_loader,
    ) = make_loaders(seed)

    model = build_swin_tiny(
        num_classes=
            len(KNOWN_CLASSES),
        dropout=
            CONFIG["dropout"],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CONFIG[
            "learning_rate"
        ],
        weight_decay=CONFIG[
            "weight_decay"
        ],
    )

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=CONFIG[
                "lr_patience"
            ],
        )
    )

    scaler = (
        torch.cuda.amp
        .GradScaler(
            enabled=(
                DEVICE.type
                == "cuda"
            )
        )
    )

    seed_dir = (
        OUTPUT_DIR /
        f"seed_{seed}"
    )

    seed_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    seed_table_dir = (
        seed_dir /
        "tables"
    )

    seed_pred_dir = (
        seed_dir /
        "predictions"
    )

    seed_fig_dir = (
        seed_dir /
        "figures"
    )

    for p in [
        seed_table_dir,
        seed_pred_dir,
        seed_fig_dir,
    ]:
        p.mkdir(
            parents=True,
            exist_ok=True,
        )

    checkpoint_path = (
        CHECKPOINT_DIR /
        f"swin_tiny_weighted_bce_seed_{seed}.pth"
    )

    history = []

    best_val_map = -np.inf
    best_epoch = None
    epochs_without_improvement = 0

    start_time = time.time()

    for epoch in range(
        1,
        CONFIG["epochs"] + 1,
    ):
        print(
            f"\nSeed {seed} | "
            f"Epoch {epoch}/"
            f"{CONFIG['epochs']}"
        )

        train_metrics = (
            train_one_epoch(
                model,
                train_loader,
                optimizer,
                scaler,
            )
        )

        val_result = (
            evaluate_loader(
                model,
                val_loader,
                calculate_loss=True,
            )
        )

        val_metrics = (
            val_result["metrics"]
        )

        scheduler.step(
            val_metrics["mAP"]
        )

        current_lr = (
            optimizer
            .param_groups[0]["lr"]
        )

        history.append({
            "seed":
                seed,
            "epoch":
                epoch,
            "lr":
                current_lr,

            "train_loss":
                train_metrics["loss"],
            "train_macro_f1":
                train_metrics[
                    "macro_f1"
                ],
            "train_micro_f1":
                train_metrics[
                    "micro_f1"
                ],
            "train_mAP":
                train_metrics["mAP"],
            "train_macro_auroc":
                train_metrics[
                    "macro_auroc"
                ],

            "val_loss":
                val_metrics["loss"],
            "val_macro_f1":
                val_metrics[
                    "macro_f1"
                ],
            "val_micro_f1":
                val_metrics[
                    "micro_f1"
                ],
            "val_mAP":
                val_metrics["mAP"],
            "val_macro_auroc":
                val_metrics[
                    "macro_auroc"
                ],
        })

        print(
            f"Train: "
            f"loss={train_metrics['loss']:.4f} | "
            f"mAP={train_metrics['mAP']:.4f} | "
            f"Macro-F1={train_metrics['macro_f1']:.4f}"
        )

        print(
            f"Val  : "
            f"loss={val_metrics['loss']:.4f} | "
            f"mAP={val_metrics['mAP']:.4f} | "
            f"Macro-F1={val_metrics['macro_f1']:.4f} | "
            f"AUROC={val_metrics['macro_auroc']:.4f}"
        )

        if (
            val_metrics["mAP"]
            > best_val_map
        ):
            best_val_map = float(
                val_metrics["mAP"]
            )

            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save({
                "model_state_dict":
                    model.state_dict(),
                "seed":
                    seed,
                "config":
                    CONFIG,
                "known_classes":
                    KNOWN_CLASSES,
                "best_val_map":
                    best_val_map,
                "best_epoch":
                    best_epoch,
                "pos_weight":
                    pos_weight.values.tolist(),
            }, checkpoint_path)

            print(
                "✓ Best checkpoint "
                f"updated: "
                f"val mAP="
                f"{best_val_map:.4f}"
            )

        else:
            epochs_without_improvement += 1

            print(
                "No improvement: "
                f"{epochs_without_improvement}/"
                f"{CONFIG['early_stopping_patience']}"
            )

        if (
            epochs_without_improvement
            >= CONFIG[
                "early_stopping_patience"
            ]
        ):
            print(
                "Early stopping."
            )
            break

    train_minutes = (
        time.time()
        - start_time
    ) / 60.0

    history_df = pd.DataFrame(
        history
    )

    history_df.to_csv(
        seed_table_dir /
        "training_history.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Load best checkpoint
    # --------------------------------------------------------

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    model.eval()

    # --------------------------------------------------------
    # Validation logits + disease thresholds
    # --------------------------------------------------------

    val_best = evaluate_loader(
        model,
        val_loader,
        calculate_loss=False,
    )

    thresholds = optimize_thresholds(
        val_best["targets"],
        val_best["probs"],
    )

    threshold_df = pd.DataFrame({
        "Disease":
            KNOWN_CLASSES,
        "Threshold":
            thresholds,
    })

    threshold_df.to_csv(
        seed_table_dir /
        "validation_thresholds.csv",
        index=False,
    )

    val_pred_df = pd.DataFrame({
        "ID":
            val_best["ids"],
        "Group":
            val_best["groups"],
    })

    for c, disease in enumerate(
        KNOWN_CLASSES
    ):
        val_pred_df[
            f"true_{disease}"
        ] = (
            val_best[
                "targets"
            ][:, c]
            .astype(int)
        )

        val_pred_df[
            f"logit_{disease}"
        ] = (
            val_best[
                "logits"
            ][:, c]
        )

        val_pred_df[
            f"prob_{disease}"
        ] = (
            val_best[
                "probs"
            ][:, c]
        )

    val_pred_df.to_csv(
        seed_pred_dir /
        "validation_logits_predictions.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Familiar test
    # --------------------------------------------------------

    familiar = evaluate_loader(
        model,
        familiar_loader,
        calculate_loss=False,
    )

    y_true = familiar["targets"]
    y_prob = familiar["probs"]

    y_pred = (
        y_prob
        >= thresholds[None, :]
    ).astype(int)

    familiar_metrics = {
        "macro_f1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    average="macro",
                    zero_division=0,
                )
            ),
        "micro_f1":
            float(
                f1_score(
                    y_true,
                    y_pred,
                    average="micro",
                    zero_division=0,
                )
            ),
        "mAP":
            safe_map(
                y_true,
                y_prob,
            ),
        "macro_auroc":
            safe_macro_auroc(
                y_true,
                y_prob,
            ),
    }

    per_class_df = (
        build_per_class_metrics(
            y_true,
            y_prob,
            thresholds,
        )
    )

    per_class_df.to_csv(
        seed_table_dir /
        "per_class_metrics.csv",
        index=False,
    )

    familiar_pred_df = pd.DataFrame({
        "ID":
            familiar["ids"],
        "Group":
            familiar["groups"],
    })

    for c, disease in enumerate(
        KNOWN_CLASSES
    ):
        familiar_pred_df[
            f"true_{disease}"
        ] = (
            y_true[:, c]
            .astype(int)
        )

        familiar_pred_df[
            f"logit_{disease}"
        ] = (
            familiar[
                "logits"
            ][:, c]
        )

        familiar_pred_df[
            f"prob_{disease}"
        ] = y_prob[:, c]

        familiar_pred_df[
            f"pred_{disease}"
        ] = (
            y_pred[:, c]
            .astype(int)
        )

    familiar_pred_df.to_csv(
        seed_pred_dir /
        "familiar_test_logits_predictions.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Unknown-only safety baseline
    # --------------------------------------------------------

    unknown = evaluate_loader(
        model,
        unknown_loader,
        calculate_loss=False,
    )

    unknown_prob = (
        unknown["probs"]
    )

    unknown_pred = (
        unknown_prob
        >= thresholds[None, :]
    ).astype(int)

    unknown_predicted_normal = (
        unknown_pred.sum(
            axis=1
        ) == 0
    )

    rare_to_normal_count = int(
        unknown_predicted_normal
        .sum()
    )

    rare_to_normal_rate = float(
        unknown_predicted_normal
        .mean()
    )

    forced_known_rate = float(
        (
            ~unknown_predicted_normal
        ).mean()
    )

    unknown_pred_df = pd.DataFrame({
        "ID":
            unknown["ids"],
        "Group":
            unknown["groups"],
        "Predicted_Normal":
            unknown_predicted_normal
            .astype(int),
        "Num_Known_Diseases_Predicted":
            unknown_pred
            .sum(axis=1)
            .astype(int),
        "Max_Known_Probability":
            unknown_prob
            .max(axis=1),
    })

    for c, disease in enumerate(
        KNOWN_CLASSES
    ):
        unknown_pred_df[
            f"logit_{disease}"
        ] = (
            unknown[
                "logits"
            ][:, c]
        )

        unknown_pred_df[
            f"prob_{disease}"
        ] = (
            unknown_prob[:, c]
        )

        unknown_pred_df[
            f"pred_{disease}"
        ] = (
            unknown_pred[:, c]
            .astype(int)
        )

    unknown_pred_df.to_csv(
        seed_pred_dir /
        "unknown_test_logits_predictions.csv",
        index=False,
    )

    # --------------------------------------------------------
    # Generalization diagnostic
    # --------------------------------------------------------

    best_history_row = (
        history_df.loc[
            history_df[
                "val_mAP"
            ].idxmax()
        ]
    )

    train_map_at_best = float(
        best_history_row[
            "train_mAP"
        ]
    )

    val_map_at_best = float(
        best_history_row[
            "val_mAP"
        ]
    )

    gap = float(
        train_map_at_best
        - val_map_at_best
    )

    seed_summary = {
        "seed":
            seed,
        "best_epoch":
            int(best_epoch),
        "training_minutes":
            float(train_minutes),

        "best_validation_mAP":
            float(best_val_map),
        "train_mAP_at_best_val":
            train_map_at_best,
        "mAP_generalization_gap":
            gap,

        "familiar_test_images":
            int(len(y_true)),
        "macro_f1":
            familiar_metrics[
                "macro_f1"
            ],
        "micro_f1":
            familiar_metrics[
                "micro_f1"
            ],
        "mAP":
            familiar_metrics[
                "mAP"
            ],
        "macro_auroc":
            familiar_metrics[
                "macro_auroc"
            ],

        "unknown_only_test_images":
            int(len(unknown_prob)),
        "rare_to_normal_count":
            rare_to_normal_count,
        "rare_to_normal_error_rate":
            rare_to_normal_rate,
        "unknown_forced_into_known_rate":
            forced_known_rate,
    }

    all_seed_summaries.append(
        seed_summary
    )

    with open(
        seed_table_dir /
        "seed_summary.json",
        "w",
    ) as f:
        json.dump(
            seed_summary,
            f,
            indent=2,
        )

    # --------------------------------------------------------
    # Curves
    # --------------------------------------------------------

    plt.figure(
        figsize=(8, 5)
    )

    plt.plot(
        history_df["epoch"],
        history_df["train_mAP"],
        label="Train",
    )

    plt.plot(
        history_df["epoch"],
        history_df["val_mAP"],
        label="Validation",
    )

    plt.xlabel("Epoch")
    plt.ylabel("mAP")

    plt.title(
        f"Swin-Tiny Weighted BCE — "
        f"mAP — Seed {seed}"
    )

    plt.legend()
    plt.tight_layout()

    plt.savefig(
        seed_fig_dir /
        "map_curve.png",
        dpi=200,
        bbox_inches="tight",
    )

    plt.close()

    plt.figure(
        figsize=(8, 5)
    )

    plt.plot(
        history_df["epoch"],
        history_df["train_loss"],
        label="Train",
    )

    plt.plot(
        history_df["epoch"],
        history_df["val_loss"],
        label="Validation",
    )

    plt.xlabel("Epoch")
    plt.ylabel("Weighted BCE Loss")

    plt.title(
        f"Swin-Tiny Weighted BCE — "
        f"Loss — Seed {seed}"
    )

    plt.legend()
    plt.tight_layout()

    plt.savefig(
        seed_fig_dir /
        "loss_curve.png",
        dpi=200,
        bbox_inches="tight",
    )

    plt.close()

    print("\nSeed summary:")

    print(
        json.dumps(
            seed_summary,
            indent=2,
        )
    )

    # Free GPU memory before next seed.
    del model
    del optimizer
    del scheduler
    del scaler
    del train_loader
    del val_loader
    del familiar_loader
    del unknown_loader

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nAll three Weighted-BCE seeds completed.")




##############################################################################
STARTING SEED 42
##############################################################################
Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pth


100%|██████████| 108M/108M [00:00<00:00, 163MB/s] 


Loaded ImageNet pretrained Swin-Tiny weights.

Seed 42 | Epoch 1/35


Train:   0%|          | 0/54 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/18 [00:00<?, ?it/s]

Train: loss=1.0118 | mAP=0.1609 | Macro-F1=0.1898
Val  : loss=0.8120 | mAP=0.3537 | Macro-F1=0.2862 | AUROC=0.8224
✓ Best checkpoint updated: val mAP=0.3537

Seed 42 | Epoch 2/35


... (266 outputs left out)

Evaluate:   0%|          | 0/18 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/18 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/1 [00:00<?, ?it/s]


Seed summary:
{
  "seed": 2026,
  "best_epoch": 22,
  "training_minutes": 91.88389252026876,
  "best_validation_mAP": 0.716094585552522,
  "train_mAP_at_best_val": 0.977267006096052,
  "mAP_generalization_gap": 0.26117242054352996,
  "familiar_test_images": 571,
  "macro_f1": 0.6777728065390266,
  "micro_f1": 0.6977928692699491,
  "mAP": 0.7342461959296597,
  "macro_auroc": 0.9467316309009609,
  "unknown_only_test_images": 32,
  "rare_to_normal_count": 7,
  "rare_to_normal_error_rate": 0.21875,
  "unknown_forced_into_known_rate": 0.78125
}

All three Weighted-BCE seeds completed.


# 17. Multi-Seed Aggregate Results

The main loss-selection metric is:

\[
\boxed{\text{Mean validation mAP across three seeds}}
\]

We also save mean ± standard deviation for the familiar-test and safety metrics.


In [17]:
# ============================================================
# 17. Aggregate results
# ============================================================

seed_results_df = pd.DataFrame(
    all_seed_summaries
)

seed_results_df.to_csv(
    TABLE_DIR /
    "all_seed_results.csv",
    index=False,
)

display(seed_results_df)

AGGREGATE_COLUMNS = [
    "best_validation_mAP",
    "mAP_generalization_gap",
    "macro_f1",
    "micro_f1",
    "mAP",
    "macro_auroc",
    "rare_to_normal_error_rate",
    "unknown_forced_into_known_rate",
]

aggregate_rows = []

for metric in AGGREGATE_COLUMNS:

    values = (
        seed_results_df[
            metric
        ].astype(float)
    )

    aggregate_rows.append({
        "Metric":
            metric,
        "Mean":
            float(
                values.mean()
            ),
        "SD":
            float(
                values.std(
                    ddof=1
                )
            ),
        "Min":
            float(
                values.min()
            ),
        "Max":
            float(
                values.max()
            ),
    })

aggregate_df = pd.DataFrame(
    aggregate_rows
)

aggregate_df.to_csv(
    TABLE_DIR /
    "multiseed_mean_sd.csv",
    index=False,
)

display(aggregate_df)

print("\n" + "=" * 78)

print(
    "SWIN-TINY WEIGHTED-BCE "
    "MULTI-SEED CLASSIFIER SUMMARY"
)

print("=" * 78)

for _, row in (
    aggregate_df
    .iterrows()
):
    print(
        f"{row['Metric']:38s}: "
        f"{row['Mean']:.6f} "
        f"± {row['SD']:.6f}"
    )


,seed,best_epoch,training_minutes,best_validation_mAP,train_mAP_at_best_val,mAP_generalization_gap,familiar_test_images,macro_f1,micro_f1,mAP,macro_auroc,unknown_only_test_images,rare_to_normal_count,rare_to_normal_error_rate,unknown_forced_into_known_rate
0,42,27,107.149389,0.751136,0.982652,0.231516,571,0.675331,0.686620,0.752923,0.954115,32,8,0.25000,0.75000
1,123,17,74.914360,0.706178,0.936590,0.230412,571,0.658668,0.682091,0.729026,0.941173,32,6,0.18750,0.81250
2,2026,22,91.883893,0.716095,0.977267,0.261172,571,0.677773,0.697793,0.734246,0.946732,32,7,0.21875,0.78125


,Metric,Mean,SD,Min,Max
0,best_validation_mAP,0.724469,0.023620,0.706178,0.751136
1,mAP_generalization_gap,0.241033,0.017450,0.230412,0.261172
2,macro_f1,0.670591,0.010397,0.658668,0.677773
3,micro_f1,0.688834,0.008082,0.682091,0.697793
4,mAP,0.738732,0.012564,0.729026,0.752923
5,macro_auroc,0.947340,0.006493,0.941173,0.954115
6,rare_to_normal_error_rate,0.218750,0.031250,0.187500,0.250000
7,unknown_forced_into_known_rate,0.781250,0.031250,0.750000,0.812500



SWIN-TINY WEIGHTED-BCE MULTI-SEED CLASSIFIER SUMMARY
best_validation_mAP                   : 0.724469 ± 0.023620
mAP_generalization_gap                : 0.241033 ± 0.017450
macro_f1                              : 0.670591 ± 0.010397
micro_f1                              : 0.688834 ± 0.008082
mAP                                   : 0.738732 ± 0.012564
macro_auroc                           : 0.947340 ± 0.006493
rare_to_normal_error_rate             : 0.218750 ± 0.031250
unknown_forced_into_known_rate        : 0.781250 ± 0.031250


# 18. Direct Multi-Seed Loss-Selection Reference

Notebook 06 produced the following ASL three-seed aggregate result:

| Metric | ASL Mean | ASL SD |
|---|---:|---:|
| Validation mAP | 0.7323 | 0.0060 |
| Macro-F1 | 0.6788 | 0.0216 |
| Micro-F1 | 0.7036 | 0.0137 |
| Test mAP | 0.7288 | 0.0122 |
| Macro AUROC | 0.9379 | 0.0047 |
| Rare→Normal error | 0.3229 | 0.0955 |

The principal decision rule remains:

> Choose the loss with the higher **mean validation mAP**.

The test/safety metrics are secondary evidence and must not be used to tune the loss.


In [18]:
# ============================================================
# 18. Weighted BCE vs ASL aggregate comparison
# ============================================================

ASL_AGGREGATE = {
    "best_validation_mAP": {
        "Mean": 0.7323,
        "SD": 0.0060,
    },
    "macro_f1": {
        "Mean": 0.6788,
        "SD": 0.0216,
    },
    "micro_f1": {
        "Mean": 0.7036,
        "SD": 0.0137,
    },
    "mAP": {
        "Mean": 0.7288,
        "SD": 0.0122,
    },
    "macro_auroc": {
        "Mean": 0.9379,
        "SD": 0.0047,
    },
    "rare_to_normal_error_rate": {
        "Mean": 0.3229,
        "SD": 0.0955,
    },
}

comparison_rows = []

for metric, asl_values in (
    ASL_AGGREGATE.items()
):
    bce_row = (
        aggregate_df[
            aggregate_df["Metric"]
            == metric
        ]
        .iloc[0]
    )

    comparison_rows.append({
        "Metric":
            metric,
        "ASL_Mean":
            asl_values["Mean"],
        "ASL_SD":
            asl_values["SD"],
        "Weighted_BCE_Mean":
            float(
                bce_row["Mean"]
            ),
        "Weighted_BCE_SD":
            float(
                bce_row["SD"]
            ),
        "BCE_minus_ASL_Mean":
            float(
                bce_row["Mean"]
                - asl_values["Mean"]
            ),
    })

loss_comparison_df = pd.DataFrame(
    comparison_rows
)

loss_comparison_df.to_csv(
    TABLE_DIR /
    "weighted_bce_vs_asl_multiseed.csv",
    index=False,
)

display(loss_comparison_df)

bce_val_map_mean = float(
    aggregate_df.loc[
        aggregate_df["Metric"]
        == "best_validation_mAP",
        "Mean",
    ].iloc[0]
)

asl_val_map_mean = float(
    ASL_AGGREGATE[
        "best_validation_mAP"
    ]["Mean"]
)

if bce_val_map_mean > asl_val_map_mean:
    recommended_loss = "Weighted BCE"
elif bce_val_map_mean < asl_val_map_mean:
    recommended_loss = "Asymmetric Loss"
else:
    recommended_loss = "Tie"

print(
    "\nValidation-mAP-based "
    "loss recommendation:",
    recommended_loss,
)


,Metric,ASL_Mean,ASL_SD,Weighted_BCE_Mean,Weighted_BCE_SD,BCE_minus_ASL_Mean
0,best_validation_mAP,0.7323,0.0060,0.724469,0.023620,-0.007831
1,macro_f1,0.6788,0.0216,0.670591,0.010397,-0.008209
2,micro_f1,0.7036,0.0137,0.688834,0.008082,-0.014766
3,mAP,0.7288,0.0122,0.738732,0.012564,0.009932
4,macro_auroc,0.9379,0.0047,0.947340,0.006493,0.009440
5,rare_to_normal_error_rate,0.3229,0.0955,0.218750,0.031250,-0.104150



Validation-mAP-based loss recommendation: Asymmetric Loss


# 19. Save Reproducibility Metadata


In [19]:
# ============================================================
# 19. Save metadata
# ============================================================

protocol = {
    "known_classes":
        KNOWN_CLASSES,
    "unknown_classes":
        UNKNOWN_CLASSES,
    "intermediate_classes":
        INTERMEDIATE_CLASSES,

    "train_images":
        len(train_exp),
    "validation_images":
        len(val_exp),
    "familiar_test_images":
        len(test_familiar),
    "unknown_only_test_images":
        len(test_unknown),
    "mixed_test_images":
        len(test_mixed),
    "intermediate_test_images":
        len(test_intermediate),

    "selected_backbone":
        "Swin-Tiny",

    "loss_under_test":
        "Weighted BCE",

    "loss_selection_rule":
        "Higher mean validation mAP across seeds 42, 123, 2026",

    "comparison_loss":
        "Asymmetric Loss",
}

with open(
    OUTPUT_DIR /
    "config.json",
    "w",
) as f:
    json.dump(
        CONFIG,
        f,
        indent=2,
    )

with open(
    OUTPUT_DIR /
    "frozen_protocol.json",
    "w",
) as f:
    json.dump(
        protocol,
        f,
        indent=2,
    )

print(
    "Reproducibility metadata saved."
)


Reproducibility metadata saved.


# 20. Create One Complete Downloadable ZIP

Everything is automatically packaged into:

`/kaggle/working/06B_SwinTiny_WeightedBCE_MultiSeed_Classifier.zip`

The ZIP contains:

- all three best Swin-Tiny Weighted-BCE checkpoints;
- all three training histories;
- validation thresholds;
- validation logits;
- familiar-test logits;
- unknown-test logits;
- per-class metrics;
- mean ± SD results;
- direct Weighted-BCE vs ASL comparison;
- figures;
- configuration and frozen-protocol metadata.

The winning-loss checkpoints will be reused directly for the SafeFundus open-set modules.


In [20]:
# ============================================================
# 20. Create complete ZIP
# ============================================================

ZIP_PATH = (
    Path("/kaggle/working")
    / f"{NOTEBOOK_STEM}.zip"
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive(
    base_name=str(
        ZIP_PATH.with_suffix("")
    ),
    format="zip",
    root_dir=OUTPUT_DIR,
)

zip_size_mb = (
    ZIP_PATH.stat().st_size
    / (1024 ** 2)
)

print("=" * 78)
print("COMPLETE OUTPUT ZIP CREATED")
print("=" * 78)

print(ZIP_PATH)

print(
    f"ZIP size: "
    f"{zip_size_mb:.1f} MB"
)

print(
    "\nDownload this single ZIP "
    "from Kaggle Outputs."
)


COMPLETE OUTPUT ZIP CREATED
/kaggle/working/06B_SwinTiny_WeightedBCE_MultiSeed_Classifier.zip
ZIP size: 293.0 MB

Download this single ZIP from Kaggle Outputs.


# Next Step

Upload:

**`06B_SwinTiny_WeightedBCE_MultiSeed_Classifier.zip`**

I will compare Weighted BCE and ASL using the complete three-seed results.

Then we will **freeze the final classifier loss** and move directly to:

## Notebook 07 — JointEnergy Open-Set Scoring

No additional classifier retraining should be necessary after the loss is frozen.
